## Midterm Project: Parallel Data Processing System (Single-Node)

---

In [7]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# imports
import pandas as pd
import numpy as np
import time
from multiprocessing import Pool, cpu_count

### Preparations
> Chosen dataset: **Global Covid-19 Dataset** (Source: Kaggle)

In [8]:
# load dataset
df = pd.read_csv("global_covid19_data.csv")

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Replace missing values
df = df.fillna(0)

print("Dataset shape:", df.shape)
df.head()



Dataset shape: (184787, 7)


,date,country,cumulative_total_cases,daily_new_cases,active_cases,cumulative_total_deaths,daily_new_deaths
0,2020-02-15,Afghanistan,0.0,0.0,0.0,0.0,0.0
1,2020-02-16,Afghanistan,0.0,0.0,0.0,0.0,0.0
2,2020-02-17,Afghanistan,0.0,0.0,0.0,0.0,0.0
3,2020-02-18,Afghanistan,0.0,0.0,0.0,0.0,0.0
4,2020-02-19,Afghanistan,0.0,0.0,0.0,0.0,0.0


---

### Data Operations (Filtering, Aggregation, and Sorting)
> Additional notes for the operations we applied:
> * Filtering: cases per country
> * Aggregation: total/average deaths per country
> * Sorting: top countries by cases

#### Sequential Version

In [9]:
# Function for sequential
def sequential_operations(df):

    # Filtering: Philippines only
    philippines = df[df["country"] == "Philippines"]

    # Aggregation: average deaths per country
    avg_deaths = df.groupby("country")["daily_new_deaths"].mean()

    # Sorting: top countries by total cases
    top_cases = df.groupby("country")["cumulative_total_cases"].max().sort_values(ascending=False)

    return philippines, avg_deaths, top_cases

#### Measure Sequential Time

In [10]:
# Measure Sequential Time
start = time.time()

philippines_seq, avg_deaths_seq, top_cases_seq = sequential_operations(df)

end = time.time()

seq_time = end - start

print("Sequential Time:", seq_time)

Sequential Time: 0.0530850887298584


#### Parallel Version

In [11]:
# Define worker function inside
def worker(chunk):
    # Filtering: Philippines
    philippines = chunk[chunk["country"] == "Philippines"]
    # Aggregation: daily_new_deaths mean per country
    avg_deaths = chunk.groupby("country")["daily_new_deaths"].mean()
    # Cases per country for sorting
    cases = chunk.groupby("country")["cumulative_total_cases"].max()
    return philippines, avg_deaths, cases

# Function for parallel
def parallel_operations(df):

    num_workers = cpu_count()

    # Split dataframe
    chunks = np.array_split(df, num_workers)

    # Parallel pool
    with Pool(processes=num_workers) as pool:
        results = pool.map(worker, chunks)

    # Combine results

    philippines = pd.concat([r[0] for r in results])

    avg_deaths = (
        pd.concat([r[1] for r in results])
        .groupby(level=0)
        .mean()
    )

    top_cases = (
        pd.concat([r[2] for r in results])
        .groupby(level=0)
        .max()
        .sort_values(ascending=False)
    )

    return philippines, avg_deaths, top_cases


if __name__ == "__main__":
    start_time = time.time()
    philippines_par, avg_deaths_par, top_cases_par = parallel_operations(df)
    end_time = time.time()
    par_time = end_time - start_time
    print("Parallel Execution Time:", par_time)

Parallel Execution Time: 0.17210674285888672


#### Measure Parallel Time

In [12]:
# Measure parallel execution time
start_time = time.time()

philippines_par, avg_deaths_par, top_cases_par = parallel_operations(df)

end_time = time.time()

par_time = end_time - start_time
print(f"Parallel Execution Time: {par_time:.4f} seconds")

Parallel Execution Time: 0.1703 seconds


#### Comparison Table

In [13]:
comparison_table = pd.DataFrame({
    "Method": ["Sequential", "Parallel"],
    "Execution Time (seconds)": [seq_time, par_time]
})

comparison_table["Speedup"] = [1, seq_time / par_time]

print(comparison_table)

       Method  Execution Time (seconds)   Speedup
0  Sequential                  0.053085  1.000000
1    Parallel                  0.170342  0.311639


# Technical Report

---

## Parallel Data Processing System Using Python Multiprocessing

### 1. Problem Description

In this project, a parallel data processing application was developed using Python to perform analytics on a moderately large dataset. The goal of the project is to compare the execution performance of sequential and parallel implementations using shared-memory parallelism. The dataset used in this project is the Global COVID-19 dataset obtained from Kaggle, which contains more than 180,000 rows of records. The system loads the dataset from a CSV file and performs several data operations including filtering, aggregation, and sorting.

---

### 2. Parallelization Approach

Two versions of the program were implemented: sequential and parallel.

The sequential version processes the dataset using normal pandas operations. It performs filtering to get records for the Philippines, aggregation to compute the average daily deaths per country, and sorting to find the countries with the highest number of total cases.

The parallel version uses Python multiprocessing to divide the dataset into multiple chunks based on the number of CPU cores available. Each chunk is processed by a worker function running in parallel. The results from all workers are then combined to produce the final output. The multiprocessing Pool class was used to manage parallel execution, and numpy array splitting was used to divide the dataset.

---

### 3. Performance Analysis

The execution time of both versions was measured using the time module in Python.

| Method     | Execution Time (seconds) | Speedup |
| ---------- | ------------------------ | ------- |
| Sequential | 0.049537                 | 1.000000    |
| Parallel   | 0.143803                 | 0.344478    |

Based on the results, the parallel version is slower than the sequential version. This happens because the dataset size is not large enough to fully benefit from parallel processing. The overhead of creating multiple processes and combining results adds extra execution time. Parallel processing is more efficient for very large datasets or computationally heavy operations.

---

### 4. Challenges Encountered

Several challenges were encountered during the development of the project. One issue was the multiprocessing error caused by defining the worker function inside another function. This caused pickling errors, which were fixed by moving the worker function to the top level. Another challenge was handling missing values in the dataset, which required replacing NaN values with zeros. There were also issues with measuring execution time in Jupyter Notebook due to missing imports. These problems were resolved by properly organizing the code and ensuring all required modules were imported.

---

### 5. Conclusion

The project successfully demonstrated the use of shared-memory parallelism in Python using multiprocessing. Both sequential and parallel implementations were completed, and their performance was compared. Although the parallel version was slower for this dataset size, the experiment shows how parallel processing can be implemented and analyzed in data processing applications.

---